# Chapter 4 - Build the Ontology Layer

Deploy Layer 1-3 ontology artifacts over Drift source data.

## Learn
- How metadata tables capture ontology definitions
- How concrete views and abstract views support routing


In [ ]:
-- Layer 2 metadata tables (minimal teaching subset)
USE ROLE SYSADMIN;
USE WAREHOUSE SFE_SAM_SNOWMAN_WH;
USE DATABASE SNOWFLAKE_EXAMPLE;
USE SCHEMA SAM_DRIFT;

CREATE OR REPLACE TABLE ONT_CLASS (
  CLASS_NAME STRING,
  PARENT_CLASS STRING,
  IS_ABSTRACT BOOLEAN,
  DESCRIPTION STRING
) COMMENT = 'DEMO: Drift ontology classes (Expires: 2026-05-22)';

CREATE OR REPLACE TABLE ONT_RELATION_DEF (
  RELATION_NAME STRING,
  DOMAIN_CLASS STRING,
  RANGE_CLASS STRING,
  DESCRIPTION STRING
) COMMENT = 'DEMO: Drift ontology relations (Expires: 2026-05-22)';

INSERT OVERWRITE INTO ONT_CLASS VALUES
  ('Person', NULL, TRUE, 'Abstract parent for customer and employee'),
  ('Customer', 'Person', FALSE, 'Commercial customer'),
  ('Employee', 'Person', FALSE, 'Drift support and ops employee'),
  ('MediaEntity', NULL, TRUE, 'Abstract parent for content objects'),
  ('Artist', 'MediaEntity', FALSE, 'Publishing artist'),
  ('Album', 'MediaEntity', FALSE, 'Album release'),
  ('Track', 'MediaEntity', FALSE, 'Individual track'),
  ('Sale', NULL, TRUE, 'Abstract parent for commerce facts'),
  ('Invoice', 'Sale', FALSE, 'Invoice header'),
  ('InvoiceLine', 'Sale', FALSE, 'Invoice detail line');

INSERT OVERWRITE INTO ONT_RELATION_DEF VALUES
  ('supports', 'Employee', 'Customer', 'Employee supports customer'),
  ('purchased', 'Customer', 'Invoice', 'Customer has invoices'),
  ('contains_track', 'Invoice', 'Track', 'Invoice lines reference tracks'),
  ('curated_on', 'Track', 'Playlist', 'Track appears on playlist');


In [ ]:
-- Layer 2 concrete typed views
CREATE OR REPLACE VIEW V_CUSTOMER AS
SELECT CUSTOMER_ID AS ENTITY_ID, FIRST_NAME, LAST_NAME, COUNTRY, SUPPORT_REP_ID
FROM CUSTOMER;

CREATE OR REPLACE VIEW V_EMPLOYEE AS
SELECT EMPLOYEE_ID AS ENTITY_ID, FIRST_NAME, LAST_NAME, TITLE, REPORTS_TO, COUNTRY
FROM EMPLOYEE;

CREATE OR REPLACE VIEW V_TRACK AS
SELECT TRACK_ID AS ENTITY_ID, NAME, GENRE_ID, UNIT_PRICE
FROM TRACK;

CREATE OR REPLACE VIEW V_INVOICE AS
SELECT INVOICE_ID AS ENTITY_ID, CUSTOMER_ID, INVOICE_DATE, TOTAL
FROM INVOICE;

CREATE OR REPLACE VIEW V_INVOICE_LINE AS
SELECT INVOICE_LINE_ID AS ENTITY_ID, INVOICE_ID, TRACK_ID, UNIT_PRICE, QUANTITY
FROM INVOICE_LINE;


In [ ]:
-- Layer 3 abstract views
CREATE OR REPLACE VIEW VW_ONT_PERSON AS
SELECT 'CUSTOMER' AS person_type, CUSTOMER_ID AS person_id, FIRST_NAME, LAST_NAME, COUNTRY
FROM CUSTOMER
UNION ALL
SELECT 'EMPLOYEE' AS person_type, EMPLOYEE_ID AS person_id, FIRST_NAME, LAST_NAME, COUNTRY
FROM EMPLOYEE;

CREATE OR REPLACE VIEW VW_ONT_MEDIAENTITY AS
SELECT 'ARTIST' AS entity_type, ARTIST_ID AS entity_id, NAME AS entity_name
FROM ARTIST
UNION ALL
SELECT 'ALBUM' AS entity_type, ALBUM_ID AS entity_id, TITLE AS entity_name
FROM ALBUM
UNION ALL
SELECT 'TRACK' AS entity_type, TRACK_ID AS entity_id, NAME AS entity_name
FROM TRACK;

CREATE OR REPLACE VIEW VW_ONT_SALE AS
SELECT 'INVOICE' AS sale_type, INVOICE_ID AS sale_id, CUSTOMER_ID, INVOICE_DATE, TOTAL
FROM INVOICE
UNION ALL
SELECT 'INVOICE_LINE' AS sale_type, INVOICE_LINE_ID AS sale_id, NULL::NUMBER AS CUSTOMER_ID, NULL::DATE AS INVOICE_DATE, UNIT_PRICE * QUANTITY AS TOTAL
FROM INVOICE_LINE;


In [ ]:
-- Check
SELECT COUNT(*) FROM ONT_CLASS;
SELECT COUNT(*) FROM ONT_RELATION_DEF;
SELECT person_type, COUNT(*) FROM VW_ONT_PERSON GROUP BY 1 ORDER BY 1;


## Reflect
Ontology layers let you ask cross-table questions through stable abstractions.

## Next
Open [`ch05_upgrade_semantic_views.ipynb`](ch05_upgrade_semantic_views.ipynb).
